> **Public archive note.** Intermediate continuation notebook with embedded ZIPs of research code and saved text-free metrics. Outputs removed, but original base64 research archives remain embedded for reproducibility. A separate security and licensing review is needed for further redistribution. Kaggle Secrets are read at runtime; values are not included.


# IJIES VIRCON-ERT — Complete Kaggle GPU Continuation (Clean Version)

**This replaces all previous notebook cells and fixes. Do not mix with the old notebook.**

- A new Kaggle notebook should use **T4 GPU** with **Internet on**.
- Attach your original **private Vircon dataset** (the two CrisisBench TSV files).
- Under **Add-ons → Secrets**, attach `KAGGLE_USERNAME` and `KAGGLE_KEY` to the notebook.
- Run **Cell 1 → Cell 2 → Cell 3 → Cell 4** in order. Cell 5 is an optional report.
- Cell 1 embeds the **62 recovered metric records**, **40 genuine detailed runs**, and **original trainer**. It checks the exact scheduler-corrected code SHA256.
- Cell 3 restores the verified private dataset `your-username/ijies-vircon62-resume-private` and uses **extracted files**, not an imaginary retained ZIP.
- Cell 4 starts with the **58 missing experiments**, trains only one at a time, and uploads/verifies a backup after **each** completion.
- The 22 original notebook-log-only runs remain tagged as such; no fake success files are generated.

**Important:** A completed experiment is durable only after its private dataset version is independently verified. Interrupted-run optimizer checkpoints are not included in the lightweight remote backups; an interrupted run may need to restart.


In [ ]:
# CELL 1 / 5 — Recover all 62 results and the exact corrected trainer
# This notebook embeds its research source and prior evaluation results.
# It does NOT embed the original train/test TSV datasets.

import base64, io, os, json, re, sys, hashlib, zipfile, shutil
from pathlib import Path
import pandas as pd

WORK = Path('/kaggle/working')
INPUT = Path('/kaggle/input')
ROOT = WORK / 'vircon_source_lr_fixed'
GRID = WORK / 'Checkpointed_Grid_LR_Fixed'
ROOT.mkdir(parents=True, exist_ok=True)
GRID.mkdir(parents=True, exist_ok=True)

# Embedded files: corrected-trainer source, 40 genuine result folders, 62 metrics.
CODE_BYTES = base64.b64decode('...')
DETAIL_BYTES = base64.b64decode('...')
METRIC_BYTES = base64.b64decode('ZXhwZXJpbWVudCxmb2xkLHNlZWQsdmFyaWFudCxtYWNyb19mMSxhY2N1cmFjeSxiYWxhbmNlZF9hY2N1cmFjeSxuX2V2YWwsZXZpZGVuY2UsaGFzX3Blcl9ldmVudCxoYXNfcGVyX2NsYXNzLGhhc19wcmVkaWN0aW9ucwpmb2xkMS9ldmVudC1kZXZfZm9sZDFfY3Jpc2lzX3NlZWQyMDI2NjAwOCwxLDIwMjY2MDA4LGNyaXNpcywwLjU2ODU0NDA3NDc0MTk1NjksMC44MDU3MTE5ODcxMjc5MTY0LDAuNzA1Mzc2Mzk2MTYzNTY2NywyNDg2LGFyY2hpdmVkX29yX25ld19zdW1tYXJ5X3Blcl9ldmVudCxUcnVlLFRydWUsRmFsc2UKZm9sZDEvZXZlbnQtZGV2X2ZvbGQxX2Rpc3RpbGJlcnRfc2VlZDIwMjY2MDA4LDEsMjAyNjYwMDgsZGlzdGlsYmVydCwwLjQ2NzkzNjcwNTY5NTU5NDYsMC43NTEwMDU2MzE1MzY2MDQ5LDAuNTc5NTgwNjA2MTEwMzEzNywyNDg2LGFyY2hpdmVkX29yX25ld19zdW1tYXJ5X3Blcl9ldmVudCxUcnVlLFRydWUsRmFsc2UKZm9sZDEvZXZlbnQtZGV2X2ZvbGQxX2Z1bGxfc2VlZDIwMjY2MDA4LDEsMjAyNjYwMDgsZnVsbCwwLjUwMzQxMDIxODk4OTU0NzksMC43NDczODUzNTgwMDQ4MjcxLDAuNjE5MDkzNDE2MDg1MDkxNCwyNDg2LGFyY2hpdmVkX29yX25ld19zdW1tYXJ5X3Blcl9ldmVudCxUcnVlLFRydWUsRmFsc2UKZm9sZDEvZXZlbnQtZGV2X2ZvbGQxX25vX2NvbnRyYXN0aXZlX3NlZWQyMDI2NjAwOCwxLDIwMjY2MDA4LG5vX2NvbnRyYXN0aXZlLDAuNDcwNzUyMTE1ODA0NTcxOSwwLjc1MzAxNjg5NDYwOTgxNDksMC41ODUwODg0NDM0NTc2ODU4LDI0ODYsYXJjaGl2ZWRfb3JfbmV3X3N1bW1hcnlfcGVyX2V2ZW50LFRydWUsVHJ1ZSxGYWxzZQpmb2xkMS9ldmVudC1kZXZfZm9sZDFfbm9fZXZlbnRfc2VlZDIwMjY2MDA4LDEsMjAyNjYwMDgsbm9fZXZlbnQsMC40NzA5MTU4MzY0NjQ0NjUsMC43NDk3OTg4NzM2OTI2NzkxLDAuNTgyNzE5NTA4MzE4ODY0OSwyNDg2LGFyY2hpdmVkX29yX25ld19zdW1tYXJ5X3Blcl9ldmVudCxUcnVlLFRydWUsRmFsc2UKZm9sZDEvZXZlbnQtZGV2X2ZvbGQxX25vX3JlcXVlc3Rfc2VlZDIwMjY2MDA4LDEsMjAyNjYwMDgsbm9fcmVxdWVzdCwwLjQ2MjA5NDU2MTYyOTcyMzEsMC43NDI1NTgzMjY2MjkxMjMxLDAuNTczNjcwNTQzODYyMjE0NywyNDg2LGFyY2hpdmVkX29yX25ld19zdW1tYXJ5X3Blcl9ldmVudCxUcnVlLFRydWUsRmFsc2UKZm9sZDEvZXZlbnQtZGV2X2ZvbGQxX25vX3RyaWFnZV9zZWVkMjAyNjYwMDgsMSwyMDI2NjAwOCxub190cmlhZ2UsMC40NzE3MzI2MzA0NTM3NTQ3LDAuNzUzNDE5MTQ3MjI0NDU2OSwwLjU4NjUzODI3NDcwMTc5NDgsMjQ4NixhcmNoaXZlZF9vcl9uZXdfc3VtbWFyeV9wZXJfZXZlbnQsVHJ1ZSxUcnVlLEZhbHNlCmZvbGQxL2V2ZW50LWRldl9mb2xkMV90ZmlkZl9zdmNfc2VlZDIwMjY2MDA4LDEsMjAyNjYwMDgsdGZpZGZfc3ZjLDAuMzc5Mzc4MzMwNzU2NTQ4NCwwLjU2NTk2OTQyODgwMTI4NzIsMC41MTE5NzIxMTg3MTM4MjQ1LDI0ODYsYXJjaGl2ZWRfb3JfbmV3X3N1bW1hcnlfcGVyX2V2ZW50LFRydWUsVHJ1ZSxGYWxzZQpmb2xkMS9ldmVudC1kZXZfZm9sZDFfY3Jpc2lzX3NlZWQyMDI2NjAwOSwxLDIwMjY2MDA5LGNyaXNpcywwLjUzNzAxMDg1NDI3MjM0NywwLjc3NTU0MzA0MTAyOTc2NjcsMC42Nzk2MjM4ODAzNDUyMzcyLDI0ODYsYXJjaGl2ZWRfb3JfbmV3X3N1bW1hcnlfcGVyX2V2ZW50LFRydWUsVHJ1ZSxGYWxzZQpmb2xkMS9ldmVudC1kZXZfZm9sZDFfZGlzdGlsYmVydF9zZWVkMjAyNjYwMDksMSwyMDI2NjAwOSxkaXN0aWxiZXJ0LDAuNDY0MDI1Mjk0MjEwMTM0LDAuNzQxNzUzODIxMzk5ODM5MSwwLjU3OTIyODkzNTA2OTgzOTIsMjQ4NixhcmNoaXZlZF9vcl9uZXdfc3VtbWFyeV9wZXJfZXZlbnQsVHJ1ZSxUcnVlLEZhbHNlCmZvbGQxL2V2ZW50LWRldl9mb2xkMV9mdWxsX3NlZWQyMDI2NjAwOSwxLDIwMjY2MDA5LGZ1bGwsMC40NTI0Nzg3MTM4OTE2OTg2LDAuNzMyOTA0MjYzODc3NzE1MiwwLjU1OTQxNTMzNjYwNDMyODIsMjQ4NixhcmNoaXZlZF9vcl9uZXdfc3VtbWFyeV9wZXJfZXZlbnQsVHJ1ZSxUcnVlLEZhbHNlCmZvbGQxL2V2ZW50LWRldl9mb2xkMV9ub19jb250cmFzdGl2ZV9zZWVkMjAyNjYwMDksMSwyMDI2NjAwOSxub19jb250cmFzdGl2ZSwwLjQ1MTM3MjUwMzIyNDAxMjIsMC43MzE2OTc1MDYwMzM3ODkyLDAuNTYwODYyNjMzMTYzMzQ2NywyNDg2LGFyY2hpdmVkX29yX25ld19zdW1tYXJ5X3Blcl9ldmVudCxUcnVlLFRydWUsRmFsc2UKZm9sZDEvZXZlbnQtZGV2X2ZvbGQxX25vX2V2ZW50X3NlZWQyMDI2NjAwOSwxLDIwMjY2MDA5LG5vX2V2ZW50LDAuNDUzODcyNjM0OTk1NzY0NCwwLjczNDExMTAyMTcyMTY0MTIsMC41NjEzMDcyMjc1NDMxMTA2LDI0ODYsYXJjaGl2ZWRfb3JfbmV3X3N1bW1hcnlfcGVyX2V2ZW50LFRydWUsVHJ1ZSxGYWxzZQpmb2xkMS9ldmVudC1kZXZfZm9sZDFfbm9fcmVxdWVzdF9zZWVkMjAyNjYwMDksMSwyMDI2NjAwOSxub19yZXF1ZXN0LDAuNDU5MDE5NDQ5NTY3Nzk2OCwwLjczNzMyOTA0MjYzODc3NzIsMC41Njg3Mzc0ODc4MjkzODE0LDI0ODYsYXJjaGl2ZWRfb3JfbmV3X3N1bW1hcnlfcGVyX2V2ZW50LFRydWUsVHJ1ZSxGYWxzZQpmb2xkMS9ldmVudC1kZXZfZm9sZDFfbm9fdHJpYWdlX3NlZWQyMDI2NjAwOSwxLDIwMjY2MDA5LG5vX3RyaWFnZSwwLjQ1ODcwNDQzMzM4NjU4NjksMC43MzYxMjIyODQ3OTQ4NTEyLDAuNTcyMTY2MDc3MDczMzQzOCwyNDg2LGFyY2hpdmVkX29yX25ld19zdW1tYXJ5X3Blcl9ldmVudCxUcnVlLFRydWUsRmFsc2UKZm9sZDEvZXZlbnQtZGV2X2ZvbGQxX3RmaWRmX3N2Y19zZWVkMjAyNjYwMDksMSwyMDI2NjAwOSx0ZmlkZl9zdmMsMC4zNzkzNzgzMzA3NTY1NDg0LDAuNTY1OTY5NDI4ODAxMjg3MiwwLjUxMTk3MjExODcxMzgyNDUsMjQ4NixhcmNoaXZlZF9vcl9uZXdfc3VtbWFyeV9wZXJfZXZlbnQsVHJ1ZSxUcnVlLEZhbHNlCmZvbGQxL2V2ZW50LWRldl9mb2xkMV9jcmlzaXNfc2VlZDIwMjY2MDEwLDEsMjAyNjYwMTAsY3Jpc2lzLDAuNTc1MzI0NzIwOTA2Njk5LDAuODA3MzIwOTk3NTg2NDg0MywwLjcwNDgwNTc1NjQzMDM5MjMsMjQ4NixhcmNoaXZlZF9vcl9uZXdfc3VtbWFyeV9wZXJfZXZlbnQsVHJ1ZSxUcnVlLEZhbHNlCmZvbGQxL2V2ZW50LWRldl9mb2xkMV9kaXN0aWxiZXJ0X3NlZWQyMDI2NjAxMCwxLDIwMjY2MDEwLGRpc3RpbGJlcnQsMC40Njg3NjM2OTk1ODA5NTMxLDAuNzQ3Nzg3NjEwNjE5NDY5MSwwLjU3OTc1NDQ3OTY4NzY2ODYsMjQ4NixhcmNoaXZlZF9vcl9uZXdfc3VtbWFyeV9wZXJfZXZlbnQsVHJ1ZSxUcnVlLEZhbHNlCmZvbGQxL2V2ZW50LWRldl9mb2xkMV9mdWxsX3NlZWQyMDI2NjAxMCwxLDIwMjY2MDEwLGZ1bGwsMC41MDcwMzQwMjI4Njk0MDgxLDAuNzQwOTQ5MzE2MTcwNTU1MiwwLjYwODY2MjEyNjU1NTc2MDUsMjQ4NixhcmNoaXZlZF9vcl9uZXdfc3VtbWFyeV9wZXJfZXZlbnQsVHJ1ZSxUcnVlLEZhbHNlCmZvbGQxL2V2ZW50LWRldl9mb2xkMV9ub19jb250cmFzdGl2ZV9zZWVkMjAyNjYwMTAsMSwyMDI2NjAxMCxub19jb250cmFzdGl2ZSwwLjQ2MDEwMTkwMzcyODkxNzMsMC43NDEzNTE1Njg3ODUxOTcxLDAuNTcxMTg2OTYyOTI4MDkxMiwyNDg2LGFyY2hpdmVkX29yX25ld19zdW1tYXJ5X3Blcl9ldmVudCxUcnVlLFRydWUsRmFsc2UKZm9sZDEvZXZlbnQtZGV2X2ZvbGQxX25vX2V2ZW50X3NlZWQyMDI2NjAxMCwxLDIwMjY2MDEwLG5vX2V2ZW50LDAuNDk4NzAwOTI4NTM1OTExNCwwLjczOTc0MjU1ODMyNjYyOTIsMC42MDg4OTMwNjI0NTY2OTg5LDI0ODYsYXJjaGl2ZWRfb3JfbmV3X3N1bW1hcnlfcGVyX2V2ZW50LFRydWUsVHJ1ZSxGYWxzZQpmb2xkMS9ldmVudC1kZXZfZm9sZDFfbm9fcmVxdWVzdF9zZWVkMjAyNjYwMTAsMSwyMDI2NjAxMCxub19yZXF1ZXN0LDAuNTAzOTY3NzM1NDAxNDU0OCwwLjc0MDE0NDgxMDk0MTI3MTIsMC42MDY2ODIyMzk3MDc4MDE5LDI0ODYsYXJjaGl2ZWRfb3JfbmV3X3N1bW1hcnlfcGVyX2V2ZW50LFRydWUsVHJ1ZSxGYWxzZQpmb2xkMS9ldmVudC1kZXZfZm9sZDFfbm9fdHJpYWdlX3NlZWQyMDI2NjAxMCwxLDIwMjY2MDEwLG5vX3RyaWFnZSwwLjQ2NjQ3NTYxNzczNjE3MjksMC43NDY1ODA4NTI3NzU1NDMxLDAuNTc4NTUxMDU3NTM1MjMyMiwyNDg2LGFyY2hpdmVkX29yX25ld19zdW1tYXJ5X3Blcl9ldmVudCxUcnVlLFRydWUsRmFsc2UKZm9sZDEvZXZlbnQtZGV2X2ZvbGQxX3RmaWRmX3N2Y19zZWVkMjAyNjYwMTAsMSwyMDI2NjAxMCx0ZmlkZl9zdmMsMC4zNzkzNzgzMzA3NTY1NDg0LDAuNTY1OTY5NDI4ODAxMjg3MiwwLjUxMTk3MjExODcxMzgyNDUsMjQ4NixhcmNoaXZlZF9vcl9uZXdfc3VtbWFyeV9wZXJfZXZlbnQsVHJ1ZSxUcnVlLEZhbHNlCmZvbGQyL2V2ZW50LWRldl9mb2xkMl9jcmlzaXNfc2VlZDIwMjY2MDA4LDIsMjAyNjYwMDgsY3Jpc2lzLDAuNDgwMDk2MDAxMzAwNDQ4OSwwLjcyMDQwMTI1NDk1Mzc2NDgsMC41NDM2MjEwMTM1Mzk5NTk2LDI0MjI0LG5vdGVib29rX2xvZ19vbmx5LEZhbHNlLEZhbHNlLEZhbHNlCmZvbGQyL2V2ZW50LWRldl9mb2xkMl9kaXN0aWxiZXJ0X3NlZWQyMDI2NjAwOCwyLDIwMjY2MDA4LGRpc3RpbGJlcnQsMC40NDkwNDA5NDQxMzIwNTMsMC43MjA2NDg5NDMxOTY4Mjk2LDAuNTAwMjYyMDcwNzY1MjE3MSwyNDIyNCxub3RlYm9va19sb2dfb25seSxGYWxzZSxGYWxzZSxGYWxzZQpmb2xkMi9ldmVudC1kZXZfZm9sZDJfZnVsbF9zZWVkMjAyNjYwMDgsMiwyMDI2NjAwOCxmdWxsLDAuNDU4OTI5ODQzNjQxOTUzMywwLjczMjE2NjQ0NjQ5OTMzOTUsMC41MDU4ODIwMjEzMzE3NTAyLDI0MjI0LG5vdGVib29rX2xvZ19vbmx5LEZhbHNlLEZhbHNlLEZhbHNlCmZvbGQyL2V2ZW50LWRldl9mb2xkMl9ub19jb250cmFzdGl2ZV9zZWVkMjAyNjYwMDgsMiwyMDI2NjAwOCxub19jb250cmFzdGl2ZSwwLjQ1NzQ3Nzc0MDk3MTUyMTksMC43MjczNzc4MDcxMzM0MjE0LDAuNTEwOTYzMzM4MDIyNDIzNCwyNDIyNCxub3RlYm9va19sb2dfb25seSxGYWxzZSxGYWxzZSxGYWxzZQpmb2xkMi9ldmVudC1kZXZfZm9sZDJfbm9fZXZlbnRfc2VlZDIwMjY2MDA4LDIsMjAyNjYwMDgsbm9fZXZlbnQsMC40NTgxNDkwNjg4ODY0MzIyLDAuNzI5NDgzMTU3MTk5NDcxNSwwLjUwNzM2MzIxNjcyNDA4OTksMjQyMjQsbm90ZWJvb2tfbG9nX29ubHksRmFsc2UsRmFsc2UsRmFsc2UKZm9sZDIvZXZlbnQtZGV2X2ZvbGQyX25vX3JlcXVlc3Rfc2VlZDIwMjY2MDA4LDIsMjAyNjYwMDgsbm9fcmVxdWVzdCwwLjQ2MTUzMDg3MDkwNjUxMzEsMC43MzE1NDcyMjU4OTE2Nzc2LDAuNTA5ODExNjYzOTM5ODkyNiwyNDIyNCxub3RlYm9va19sb2dfb25seSxGYWxzZSxGYWxzZSxGYWxzZQpmb2xkMi9ldmVudC1kZXZfZm9sZDJfbm9fdHJpYWdlX3NlZWQyMDI2NjAwOCwyLDIwMjY2MDA4LG5vX3RyaWFnZSwwLjQ1MjQwMDI5MTE4MTc4NTUsMC43MzA1NTY0NzI5MTk0MTg3LDAuNDk0NjIzOTMxNzc3Mjk0MiwyNDIyNCxub3RlYm9va19sb2dfb25seSxGYWxzZSxGYWxzZSxGYWxzZQpmb2xkMi9ldmVudC1kZXZfZm9sZDJfdGZpZGZfc3ZjX3NlZWQyMDI2NjAwOCwyLDIwMjY2MDA4LHRmaWRmX3N2YywwLjM4NDU0NzQ4NTMzNjkzMjMsMC43MTkxNjI4MTM3Mzg0NDEzLDAuMzg0NTgyNTI2MDA2ODc3MiwyNDIyNCxub3RlYm9va19sb2dfb25seSxGYWxzZSxGYWxzZSxGYWxzZQpmb2xkMi9ldmVudC1kZXZfZm9sZDJfY3Jpc2lzX3NlZWQyMDI2NjAwOSwyLDIwMjY2MDA5LGNyaXNpcywwLjQ3OTUzNTk1NTkzMjM3NjMsMC43MjExNDQzMTk2ODI5NTksMC41MzkwMjQ4OTI5ODQ2ODc3LDI0MjI0LGFyY2hpdmVkX29yX25ld19zdW1tYXJ5X3Blcl9ldmVudCxUcnVlLFRydWUsRmFsc2UKZm9sZDIvZXZlbnQtZGV2X2ZvbGQyX2Rpc3RpbGJlcnRfc2VlZDIwMjY2MDA5LDIsMjAyNjYwMDksZGlzdGlsYmVydCwwLjQ0NDc3OTkwNDY0MTc0MTYsMC43MjUxODk4OTQzMTk2ODMsMC40OTAzNjI1MTY4NjY0ODEzLDI0MjI0LGFyY2hpdmVkX29yX25ld19zdW1tYXJ5X3Blcl9ldmVudCxUcnVlLFRydWUsRmFsc2UKZm9sZDIvZXZlbnQtZGV2X2ZvbGQyX2Z1bGxfc2VlZDIwMjY2MDA5LDIsMjAyNjYwMDksZnVsbCwwLjQ0Mjc0ODI5MTAzNTI1NjgsMC43MzEzNDA4MTkwMjI0NTcxLDAuNDgwNTE3Nzg2MzY5MzAzNSwyNDIyNCxhcmNoaXZlZF9vcl9uZXdfc3VtbWFyeV9wZXJfZXZlbnQsVHJ1ZSxUcnVlLEZhbHNlCmZvbGQyL2V2ZW50LWRldl9mb2xkMl9ub19jb250cmFzdGl2ZV9zZWVkMjAyNjYwMDksMiwyMDI2NjAwOSxub19jb250cmFzdGl2ZSwwLjQ0MDA4MDQ1MTg1MjQwODksMC43MjY1NTIxNzk2NTY1MzksMC40ODM4MDUyNzg4OTk0NjA0LDI0MjI0LGFyY2hpdmVkX29yX25ld19zdW1tYXJ5X3Blcl9ldmVudCxUcnVlLFRydWUsRmFsc2UKZm9sZDIvZXZlbnQtZGV2X2ZvbGQyX25vX2V2ZW50X3NlZWQyMDI2NjAwOSwyLDIwMjY2MDA5LG5vX2V2ZW50LDAuNDQxNTAyODAzNjgzOTQ0NiwwLjcyOTczMDg0NTQ0MjUzNjMsMC40ODAzNzUxNjU4NTc0NDY1LDI0MjI0LGFyY2hpdmVkX29yX25ld19zdW1tYXJ5X3Blcl9ldmVudCxUcnVlLFRydWUsRmFsc2UKZm9sZDIvZXZlbnQtZGV2X2ZvbGQyX25vX3JlcXVlc3Rfc2VlZDIwMjY2MDA5LDIsMjAyNjYwMDksbm9fcmVxdWVzdCwwLjQzODE5NDQ5MDQ3MzA3NTYsMC43Mjk2MDcwMDEzMjEwMDQsMC40NzY5NDM3ODA2ODIzMjYsMjQyMjQsYXJjaGl2ZWRfb3JfbmV3X3N1bW1hcnlfcGVyX2V2ZW50LFRydWUsVHJ1ZSxGYWxzZQpmb2xkMi9ldmVudC1kZXZfZm9sZDJfbm9fdHJpYWdlX3NlZWQyMDI2NjAwOSwyLDIwMjY2MDA5LG5vX3RyaWFnZSwwLjQ0ODEyMjE4OTA0OTk5MjYsMC43MzAwMTk4MTUwNTk0NDUxLDAuNDg2NDEwMjgwOTUwOTUxNCwyNDIyNCxhcmNoaXZlZF9vcl9uZXdfc3VtbWFyeV9wZXJfZXZlbnQsVHJ1ZSxUcnVlLEZhbHNlCmZvbGQyL2V2ZW50LWRldl9mb2xkMl90ZmlkZl9zdmNfc2VlZDIwMjY2MDA5LDIsMjAyNjYwMDksdGZpZGZfc3ZjLDAuMzg0NTQ3NDg1MzM2OTMyMywwLjcxOTE2MjgxMzczODQ0MTMsMC4zODQ1ODI1MjYwMDY4NzcyLDI0MjI0LGFyY2hpdmVkX29yX25ld19zdW1tYXJ5X3Blcl9ldmVudCxUcnVlLFRydWUsRmFsc2UKZm9sZDIvZXZlbnQtZGV2X2ZvbGQyX2NyaXNpc19zZWVkMjAyNjYwMTAsMiwyMDI2NjAxMCxjcmlzaXMsMC40NzM0MTc1NjAzNjIxMjIsMC43MTUxOTk4MDE4NDk0MDU2LDAuNTM5NzAyMjg1NTIwNTE3MSwyNDIyNCxub3RlYm9va19sb2dfb25seSxGYWxzZSxGYWxzZSxGYWxzZQpmb2xkMi9ldmVudC1kZXZfZm9sZDJfZGlzdGlsYmVydF9zZWVkMjAyNjYwMTAsMiwyMDI2NjAxMCxkaXN0aWxiZXJ0LDAuNDUzMjkzNDI2OTE4NjQ0NiwwLjcyMjIxNzYzNTQwMjkwNjIsMC41MTI0Mjg5MjU3NjE2MjI4LDI0MjI0LG5vdGVib29rX2xvZ19vbmx5LEZhbHNlLEZhbHNlLEZhbHNlCmZvbGQyL2V2ZW50LWRldl9mb2xkMl9mdWxsX3NlZWQyMDI2NjAxMCwyLDIwMjY2MDEwLGZ1bGwsMC40NjAxMTgwOTQ4MjEzNDg5LDAuNzMyOTA5NTExMjI4NTMzNiwwLjUwNjc4Mjc1MjIwOTY4MTYsMjQyMjQsYXJjaGl2ZWRfb3JfbmV3X3N1bW1hcnlfcGVyX2V2ZW50LFRydWUsVHJ1ZSxGYWxzZQpmb2xkMi9ldmVudC1kZXZfZm9sZDJfbm9fY29udHJhc3RpdmVfc2VlZDIwMjY2MDEwLDIsMjAyNjYwMTAsbm9fY29udHJhc3RpdmUsMC40NTgyOTYyOTAwNDI3NDk4LDAuNzI5NDAwNTk0NDUxNzgzNCwwLjUxMDU1NzA0NDMxNDk4NTYsMjQyMjQsYXJjaGl2ZWRfb3JfbmV3X3N1bW1hcnlfcGVyX2V2ZW50LFRydWUsVHJ1ZSxGYWxzZQpmb2xkMi9ldmVudC1kZXZfZm9sZDJfbm9fZXZlbnRfc2VlZDIwMjY2MDEwLDIsMjAyNjYwMTAsbm9fZXZlbnQsMC40NTg1MjIyMDg5MTI5MzExLDAuNzMwNDMyNjI4Nzk3ODg2NCwwLjUwNjE1MjIyMTEwNTk3NDcsMjQyMjQsYXJjaGl2ZWRfb3JfbmV3X3N1bW1hcnlfcGVyX2V2ZW50LFRydWUsVHJ1ZSxGYWxzZQpmb2xkMi9ldmVudC1kZXZfZm9sZDJfbm9fcmVxdWVzdF9zZWVkMjAyNjYwMTAsMiwyMDI2NjAxMCxub19yZXF1ZXN0LDAuNDU2NzE3ODU1MDA4MTA0LDAuNzMwMzkxMzQ3NDI0MDQyMywwLjUwNTAyNTU4NTU2MDQxNTUsMjQyMjQsbm90ZWJvb2tfbG9nX29ubHksRmFsc2UsRmFsc2UsRmFsc2UKZm9sZDIvZXZlbnQtZGV2X2ZvbGQyX25vX3RyaWFnZV9zZWVkMjAyNjYwMTAsMiwyMDI2NjAxMCxub190cmlhZ2UsMC40NTgyOTc3MTA2NzQ1NjAxLDAuNzMxNDY0NjYzMTQzOTg5NCwwLjUwNDY3ODYxOTMwNzEzMjYsMjQyMjQsbm90ZWJvb2tfbG9nX29ubHksRmFsc2UsRmFsc2UsRmFsc2UKZm9sZDIvZXZlbnQtZGV2X2ZvbGQyX3RmaWRmX3N2Y19zZWVkMjAyNjYwMTAsMiwyMDI2NjAxMCx0ZmlkZl9zdmMsMC4zODQ1NDc0ODUzMzY5MzIzLDAuNzE5MTYyODEzNzM4NDQxMywwLjM4NDU4MjUyNjAwNjg3NzIsMjQyMjQsbm90ZWJvb2tfbG9nX29ubHksRmFsc2UsRmFsc2UsRmFsc2UKZm9sZDMvZXZlbnQtZGV2X2ZvbGQzX2NyaXNpc19zZWVkMjAyNjYwMDksMywyMDI2NjAwOSxjcmlzaXMsMC40MTQ1MzAyODc3Mjk4ODg3LDAuNjMyOTc5NDg2NDQzODM4NywwLjUzOTE5MzMzMjE1ODUyMjcsMTM5NDIsYXJjaGl2ZWRfb3JfbmV3X3N1bW1hcnlfcGVyX2V2ZW50LFRydWUsVHJ1ZSxUcnVlCmZvbGQzL2V2ZW50LWRldl9mb2xkM19kaXN0aWxiZXJ0X3NlZWQyMDI2NjAwOSwzLDIwMjY2MDA5LGRpc3RpbGJlcnQsMC4zOTc4NjU0NTExNTA1NjA1LDAuNjQ2NjA3MzczNDA0MTAyNywwLjUwMzYxNTM4MDYyNTI0MTksMTM5NDIsYXJjaGl2ZWRfb3JfbmV3X3N1bW1hcnlfcGVyX2V2ZW50LFRydWUsVHJ1ZSxUcnVlCmZvbGQzL2V2ZW50LWRldl9mb2xkM19mdWxsX3NlZWQyMDI2NjAwOSwzLDIwMjY2MDA5LGZ1bGwsMC40MDIyODY1MzYzMDg3NDA3LDAuNjUyMTMwMjUzOTA5MDUxOCwwLjUwODUwMzU3MjQ4ODc0NzIsMTM5NDIsbm90ZWJvb2tfbG9nX29ubHksRmFsc2UsRmFsc2UsRmFsc2UKZm9sZDMvZXZlbnQtZGV2X2ZvbGQzX25vX2NvbnRyYXN0aXZlX3NlZWQyMDI2NjAwOSwzLDIwMjY2MDA5LG5vX2NvbnRyYXN0aXZlLDAuNDAxNjU1MTk5ODI0Mjc4NSwwLjY0NDQ1NTYwMTc3ODc5NzksMC41MTEzMzU3NzY5MDI3OTMzLDEzOTQyLGFyY2hpdmVkX29yX25ld19zdW1tYXJ5X3Blcl9ldmVudCxUcnVlLFRydWUsVHJ1ZQpmb2xkMy9ldmVudC1kZXZfZm9sZDNfbm9fZXZlbnRfc2VlZDIwMjY2MDA5LDMsMjAyNjYwMDksbm9fZXZlbnQsMC40MDI1Mzc2MDU4NDQ1MzY5LDAuNjUwNjI0MDEzNzcxMzM4NCwwLjUwOTYyNjYxMDAyNzYxMTEsMTM5NDIsbm90ZWJvb2tfbG9nX29ubHksRmFsc2UsRmFsc2UsRmFsc2UKZm9sZDMvZXZlbnQtZGV2X2ZvbGQzX25vX3JlcXVlc3Rfc2VlZDIwMjY2MDA5LDMsMjAyNjYwMDksbm9fcmVxdWVzdCwwLjQwMjQ2NzAxMzkwODI2MTQsMC42NTM2MzY0OTQwNDY3NjUyLDAuNTA2MTk5NDU3MTMzNzcyMSwxMzk0MixhcmNoaXZlZF9vcl9uZXdfc3VtbWFyeV9wZXJfZXZlbnQsVHJ1ZSxUcnVlLFRydWUKZm9sZDMvZXZlbnQtZGV2X2ZvbGQzX25vX3RyaWFnZV9zZWVkMjAyNjYwMDksMywyMDI2NjAwOSxub190cmlhZ2UsMC40MDAyNDAxNDI3OTgwMjYzLDAuNjUyMjAxOTc5NjI5ODk1MiwwLjUwMzE0NjA3NTczODgyNzMsMTM5NDIsYXJjaGl2ZWRfb3JfbmV3X3N1bW1hcnlfcGVyX2V2ZW50LFRydWUsVHJ1ZSxUcnVlCmZvbGQzL2V2ZW50LWRldl9mb2xkM190ZmlkZl9zdmNfc2VlZDIwMjY2MDA5LDMsMjAyNjYwMDksdGZpZGZfc3ZjLDAuMzcxODIyMDAwMDE0MjIwNjcsMC42ODg3MTAzNzE1MzkyMzQsMC40MjYyMzExMjc0MjEzMTEzNiwxMzk0Mixub3RlYm9va19sb2dfb25seSxGYWxzZSxGYWxzZSxGYWxzZQpmb2xkMy9ldmVudC1kZXZfZm9sZDNfZGlzdGlsYmVydF9zZWVkMjAyNjYwMTAsMywyMDI2NjAxMCxkaXN0aWxiZXJ0LDAuNDA3ODMyNjMwMTA4MjM3MiwwLjY0MTA4NDQ5Mjg5OTE1MzYsMC41MTcyNDY1NjM0NjkwODMyLDEzOTQyLG5vdGVib29rX2xvZ19vbmx5LEZhbHNlLEZhbHNlLEZhbHNlCmZvbGQzL2V2ZW50LWRldl9mb2xkM19mdWxsX3NlZWQyMDI2NjAxMCwzLDIwMjY2MDEwLGZ1bGwsMC40MDc4MDI1NzUzMDAyNjA5LDAuNjQ0ODE0MjMwMzgzMDE1MywwLjUxODg1OTgwNjM5MDc0MywxMzk0Mixub3RlYm9va19sb2dfb25seSxGYWxzZSxGYWxzZSxGYWxzZQpmb2xkMy9ldmVudC1kZXZfZm9sZDNfbm9fY29udHJhc3RpdmVfc2VlZDIwMjY2MDEwLDMsMjAyNjYwMTAsbm9fY29udHJhc3RpdmUsMC40MDQyMjczNTQ2ODkxNjE2LDAuNjQzMDIxMDg3MzYxOTI4LDAuNTE4NzM3MzkxNjI1MDgxNiwxMzk0Mixub3RlYm9va19sb2dfb25seSxGYWxzZSxGYWxzZSxGYWxzZQpmb2xkMy9ldmVudC1kZXZfZm9sZDNfbm9fZXZlbnRfc2VlZDIwMjY2MDEwLDMsMjAyNjYwMTAsbm9fZXZlbnQsMC40MDYyOTQ2MDA5ODQ3OTgyLDAuNjQwODY5MzE1NzM2NjIzMiwwLjUxODQxMTc0MTE5OTU3NSwxMzk0Mixub3RlYm9va19sb2dfb25seSxGYWxzZSxGYWxzZSxGYWxzZQpmb2xkMy9ldmVudC1kZXZfZm9sZDNfbm9fcmVxdWVzdF9zZWVkMjAyNjYwMTAsMywyMDI2NjAxMCxub19yZXF1ZXN0LDAuNDAzNTk2MzUzMDUwNjQxNiwwLjY0MzA5MjgxMzA4Mjc3MTQsMC41MTQwODA0NDczNTMyNzgsMTM5NDIsbm90ZWJvb2tfbG9nX29ubHksRmFsc2UsRmFsc2UsRmFsc2UKZm9sZDMvZXZlbnQtZGV2X2ZvbGQzX25vX3RyaWFnZV9zZWVkMjAyNjYwMTAsMywyMDI2NjAxMCxub190cmlhZ2UsMC40MDU3MzgwOTQyNDk0MzQzNiwwLjY0Mzg4MTc5NjAxMjA0OTksMC41MTAxODE4ODEyNTgzOTAxLDEzOTQyLG5vdGVib29rX2xvZ19vbmx5LEZhbHNlLEZhbHNlLEZhbHNlCg==')

# Extract only regular files within the expected source archive directory.
with zipfile.ZipFile(io.BytesIO(CODE_BYTES)) as archive:
    prefix = 'VIRCON_ERT_IJIES_Round2_Resumable/'
    for member in archive.infolist():
        if member.is_dir() or not member.filename.startswith(prefix):
            continue
        rel = Path(member.filename[len(prefix):])
        if not rel.parts or rel.is_absolute() or '..' in rel.parts:
            continue
        dst = ROOT / rel
        if dst.is_file():
            continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member) as src, dst.open('wb') as target:
            shutil.copyfileobj(src, target)

TRAINER = ROOT / 'scripts/run_experiment_resumable.py'
GRID_SCRIPT = ROOT / 'scripts/stage2_checkpoint_grid.py'
FOLDS = ROOT / 'data_provenance/event_disjoint_fold_assignment.csv'
assert all(p.is_file() for p in (TRAINER, GRID_SCRIPT, FOLDS)), 'Missing source files'

# Preserve the scheduler fix used by every recovered corrected-grid run.
old = ('            scaler.step(optimizer)\n'
       '            scaler.update()\n'
       '            scheduler.step()')
new = ('            scale_before = scaler.get_scale()\n'
       '            scaler.step(optimizer)\n'
       '            scaler.update()\n'
       '            scale_after = scaler.get_scale()\n'
       '            if scale_after >= scale_before:\n'
       '                scheduler.step()')
source = TRAINER.read_text()
if old in source:
    assert source.count(old) == 1
    TRAINER.write_text(source.replace(old, new))
elif new not in source:
    raise RuntimeError('Unknown trainer version: refusing to change the protocol.')

EXPECTED_HASH = 'ff5783a9aa4fc9f3d25ba11834d1cb86b8b65296d214f9ec3c35c374543bdda1'
actual_hash = hashlib.sha256(TRAINER.read_bytes()).hexdigest()
assert actual_hash == EXPECTED_HASH, f'Corrected trainer checksum mismatch: {actual_hash}'

# Restore genuine 40 detailed completed-run folders. No fabricated SUCCESS records.
with zipfile.ZipFile(io.BytesIO(DETAIL_BYTES)) as archive:
    for member in archive.infolist():
        if member.is_dir():
            continue
        rel = Path(member.filename)
        if (len(rel.parts) < 3 or rel.is_absolute() or '..' in rel.parts
            or not re.fullmatch(r'fold[1-5]', rel.parts[0])
            or not rel.parts[1].startswith('event-dev_fold')):
            continue
        dst = GRID / rel
        if dst.exists():
            continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member) as src, dst.open('wb') as target:
            shutil.copyfileobj(src, target)

METRICS_PATH = WORK / 'IJIES_Recovery_Combined_Metrics.csv'
if not METRICS_PATH.exists():
    METRICS_PATH.write_bytes(METRIC_BYTES)

SEEDS = [20266009, 20266010, 20266008]
VARIANTS = ['full', 'no_event', 'no_contrastive', 'no_request', 'no_triage',
            'distilbert', 'crisis', 'tfidf_svc']
ALL_RUNS = [
    (fold, seed, variant, f'fold{fold}/event-dev_fold{fold}_{variant}_seed{seed}')
    for fold in range(1, 6) for seed in SEEDS for variant in VARIANTS
]
CANONICAL = {row[3] for row in ALL_RUNS}
assert len(CANONICAL) == 120


def completed_artifact_runs():
    done = set()
    for success in GRID.glob('fold*/event-dev_fold*/SUCCESS.json'):
        try:
            payload = json.loads(success.read_text())
            if payload.get('completed') is True and (success.parent / 'summary.json').is_file():
                done.add(success.parent.relative_to(GRID).as_posix())
        except (OSError, ValueError):
            continue
    return done


def load_metrics():
    df = pd.read_csv(METRICS_PATH)
    assert 'experiment' in df.columns, 'Missing experiment column'
    df['experiment'] = df['experiment'].astype(str)
    assert df.experiment.is_unique, 'Duplicate experiment IDs'
    assert set(df.experiment).issubset(CANONICAL), 'Unknown experiment ID'
    return df


def pending(retrain_log_only=False):
    skip = completed_artifact_runs()
    if not retrain_log_only:
        skip = skip | set(load_metrics().experiment)
    return [row for row in ALL_RUNS if row[3] not in skip]

print('RECOVERY READY')
print('Metric records:', len(load_metrics()))
print('Genuine detailed runs:', len(completed_artifact_runs()))
print('Missing experiments:', len(pending()))
print('Corrected trainer SHA256:', actual_hash)
assert len(load_metrics()) >= 62 and len(completed_artifact_runs()) >= 40


In [ ]:
# CELL 2 / 5 — Attach your original Vircon TSV input, GPU, and Kaggle secrets
# Kaggle notebook settings: Accelerator = T4 GPU; Internet = On.
# Kaggle > Add-ons > Secrets: attach KAGGLE_USERNAME and KAGGLE_KEY.

import os, sys, re, shutil, subprocess, hashlib
from pathlib import Path
import torch
from kaggle_secrets import UserSecretsClient

TRAIN_NAME = 'crisis_consolidated_humanitarian_filtered_lang_en_train.tsv'
TEST_NAME = 'crisis_consolidated_humanitarian_filtered_lang_en_test.tsv'


def find_data_file(filename, expected_sha):
    candidates = list(INPUT.rglob(filename))
    if not candidates:
        raise FileNotFoundError('Add your original private Vircon Kaggle input: ' + filename)
    matches = []
    for candidate in candidates:
        h = hashlib.sha256()
        with candidate.open('rb') as f:
            for block in iter(lambda: f.read(1024 * 1024), b''):
                h.update(block)
        if h.hexdigest() == expected_sha:
            matches.append(candidate)
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one checksum-matching {filename}; found {len(matches)}.')
    return matches[0]

TRAIN = find_data_file(TRAIN_NAME, '5e3dd75d1dcbdba893d8d731795b51853d04708547a9bdc2bd18a0b54b82541b')
TEST = find_data_file(TEST_NAME, 'a664ce57f40c8791228d5a00ab43c23965c2f1be90fde4307413217fca8d1a86')

assert torch.cuda.is_available(), 'In Kaggle settings, enable T4 GPU before continuing.'
print('GPU:', torch.cuda.get_device_name(0))

# Install only the original experiment requirements; preserve CUDA-enabled torch.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements-colab.txt')])

# Install/upgrade Kaggle CLI to a tested version (not python -m kaggle).
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'kaggle==2.2.5'])
KAGGLE_CLI = shutil.which('kaggle')
assert KAGGLE_CLI, 'Cannot find Kaggle CLI executable.'

secrets = UserSecretsClient()
username = secrets.get_secret('KAGGLE_USERNAME').strip()
key = secrets.get_secret('KAGGLE_KEY').strip()
assert re.fullmatch(r'[A-Za-z0-9_-]+', username)
assert len(key) > 10
os.environ['KAGGLE_USERNAME'] = username
os.environ['KAGGLE_KEY'] = key
# Do not print username/key. Secrets are only passed as process environment variables.

DATASET_REF = username + '/ijies-vircon62-resume-private'
BACKUP_ZIP = WORK / 'IJIES_Persistent_Backup.zip'


def run_kaggle(args, timeout=900):
    return subprocess.run(
        [KAGGLE_CLI] + list(args), capture_output=True, text=True,
        timeout=timeout, env=os.environ.copy()
    )

print('TRAIN and TEST fingerprints: VERIFIED')
print('Kaggle secrets: ACCESS OK')
print('Backup destination:', DATASET_REF)


In [ ]:
# CELL 3 / 5 — Restore existing private backup and define WORKING backup functions
# No "datasets status" calls and NO search for an already extracted ZIP.
# The user's existing dataset is read as extracted grid/ files + metrics CSV.

import io, os, json, time, shutil, zipfile, hashlib
import pandas as pd
from pathlib import Path

BACKUP_KEEP = {
    'SUCCESS.json', 'summary.json', 'triage_summary.json', 'per_event.csv',
    'per_class.csv', 'epoch_history.csv', 'run_manifest.json',
    'model_structure.json', 'train_event_class_support.csv',
    'eval_event_class_support.csv', 'predictions_no_text.csv',
    'completed_grid_summary.csv', 'training_log.txt',
}


def download_remote_dataset(destination):
    destination = Path(destination)
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)
    result = run_kaggle([
        'datasets', 'download', DATASET_REF,
        '-p', str(destination), '--unzip', '-o', '-q'
    ], timeout=900)
    if result.returncode:
        raise RuntimeError(
            'Remote private backup download failed. No new training was started.\n'
            + (result.stderr + result.stdout)[-1500:]
        )
    candidates = list(destination.rglob('IJIES_Recovery_Combined_Metrics.csv'))
    if len(candidates) != 1:
        visible = [str(p.relative_to(destination)) for p in destination.rglob('*') if p.is_file()]
        raise RuntimeError(f'Expected one remote metrics CSV, got {len(candidates)}. Files: {visible[:12]}')
    root = candidates[0].parent
    if not (root / 'grid').is_dir():
        raise RuntimeError('Remote backup is missing its grid/ directory.')
    return root


def merge_tables(local, remote):
    metric_keys = ['macro_f1', 'accuracy', 'balanced_accuracy']
    records = {str(r['experiment']): dict(r) for r in local.to_dict('records')}
    for row in remote.to_dict('records'):
        name = str(row['experiment'])
        if name in records:
            for col in metric_keys:
                a = pd.to_numeric(records[name].get(col), errors='coerce')
                b = pd.to_numeric(row.get(col), errors='coerce')
                if pd.notna(a) and pd.notna(b) and abs(float(a) - float(b)) > 1e-8:
                    raise RuntimeError(f'Conflicting {col} for {name}; stopping without overwriting.')
            # Prefer a detailed artifact record to a log-only record.
            if 'genuine' in str(row.get('evidence', '')) or 'archived' in str(row.get('evidence', '')):
                records[name] = dict(row)
        else:
            records[name] = dict(row)
    result = pd.DataFrame(list(records.values())).sort_values('experiment').reset_index(drop=True)
    if result.experiment.duplicated().any() or not set(result.experiment).issubset(CANONICAL):
        raise RuntimeError('Unexpected experiment identifiers in the private backup.')
    return result


def import_latest_backup():
    print('Restoring private Kaggle results...')
    remote_root = download_remote_dataset(WORK / 'ijies_private_restore')
    remote_metrics = pd.read_csv(remote_root / 'IJIES_Recovery_Combined_Metrics.csv')
    local_metrics = load_metrics()
    merged = merge_tables(local_metrics, remote_metrics)
    copied = 0
    for src in (remote_root / 'grid').rglob('*'):
        if not src.is_file():
            continue
        rel = src.relative_to(remote_root / 'grid')
        dst = GRID / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists():
            if hashlib.sha256(src.read_bytes()).digest() != hashlib.sha256(dst.read_bytes()).digest():
                raise RuntimeError(f'Conflicting local and remote detailed artifact: {rel}')
        else:
            shutil.copy2(src, dst)
            copied += 1
    merged.to_csv(METRICS_PATH, index=False)
    print('Remote restored:', len(merged), 'metric records;', copied, 'artifact files added.')
    return len(merged)


def merge_completed_metrics():
    records = {str(r['experiment']): dict(r) for r in load_metrics().to_dict('records')}
    for name in sorted(completed_artifact_runs()):
        folder = GRID / name
        summary = json.loads((folder / 'summary.json').read_text())
        match = re.fullmatch(r'event-dev_fold(\d+)_(.+)_seed(\d+)', folder.name)
        if not match:
            raise RuntimeError(f'Unexpected run folder name: {name}')
        fold, variant, seed = match.groups()
        row = records.get(name, {})
        row.update({
            'experiment': name, 'fold': int(fold), 'seed': int(seed), 'variant': variant,
            'macro_f1': summary.get('macro_f1'), 'accuracy': summary.get('accuracy'),
            'balanced_accuracy': summary.get('balanced_accuracy'),
            'n_eval': summary.get('n', summary.get('n_eval')),
            'evidence': 'genuine_completed_artifact',
            'has_per_event': (folder / 'per_event.csv').is_file(),
            'has_per_class': (folder / 'per_class.csv').is_file(),
            'has_predictions': (folder / 'predictions_no_text.csv').is_file(),
        })
        records[name] = row
    result = pd.DataFrame(list(records.values())).sort_values('experiment')
    if not set(result.experiment).issubset(CANONICAL):
        raise RuntimeError('Noncanonical experiment found.')
    result.to_csv(METRICS_PATH, index=False)
    return len(result)


def build_snapshot():
    rows = merge_completed_metrics()
    temporary = WORK / 'IJIES_Persistent_Backup.tmp'
    with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_DEFLATED, compresslevel=1) as z:
        z.write(METRICS_PATH, 'IJIES_Recovery_Combined_Metrics.csv')
        z.writestr('backup_manifest.json', json.dumps({
            'total_metric_records': rows,
            'genuine_artifact_runs': len(completed_artifact_runs()),
            'source_checksum': EXPECTED_HASH,
            'note': 'Log-only records do not contain predictions/per-event artifacts.',
        }, indent=2))
        for path in sorted(GRID.rglob('*')):
            if path.is_file() and path.name in BACKUP_KEEP:
                z.write(path, 'grid/' + path.relative_to(GRID).as_posix())
    os.replace(temporary, BACKUP_ZIP)
    return rows


def verify_uploaded_snapshot(expected_rows):
    remote_root = download_remote_dataset(WORK / 'ijies_private_verification')
    remote_df = pd.read_csv(remote_root / 'IJIES_Recovery_Combined_Metrics.csv')
    if len(remote_df) != expected_rows:
        raise RuntimeError(f'Remote count {len(remote_df)}, expected {expected_rows}.')
    # Confirm every file inside the local snapshot exactly matches the remote extracted file.
    checked = 0
    with zipfile.ZipFile(BACKUP_ZIP) as z:
        for entry in z.infolist():
            if entry.is_dir():
                continue
            remote_file = remote_root / entry.filename
            if not remote_file.is_file():
                raise RuntimeError('Remote backup file missing: ' + entry.filename)
            local_hash = hashlib.sha256()
            remote_hash = hashlib.sha256()
            with z.open(entry) as f:
                for piece in iter(lambda: f.read(1024 * 1024), b''):
                    local_hash.update(piece)
            with remote_file.open('rb') as f:
                for piece in iter(lambda: f.read(1024 * 1024), b''):
                    remote_hash.update(piece)
            if local_hash.digest() != remote_hash.digest():
                raise RuntimeError('Remote content mismatch: ' + entry.filename)
            checked += 1
    print(f'REMOTE BACKUP VERIFIED: {expected_rows}/120 records; {checked} files checked.')
    return True


def push_private_backup(message):
    rows = build_snapshot()
    stage = WORK / 'ijies_backup_stage'
    if stage.exists():
        shutil.rmtree(stage)
    stage.mkdir(parents=True, exist_ok=True)
    shutil.copy2(BACKUP_ZIP, stage / BACKUP_ZIP.name)
    metadata = {
        'title': 'IJIES VIRCON Experiment Recovery',
        'id': DATASET_REF,
        'licenses': [{'name': 'unknown'}],
        'subtitle': 'Private recovered experiment results',
        'description': 'Private VIRCON-ERT evaluation metrics and completed-run artifacts.'
    }
    (stage / 'dataset-metadata.json').write_text(json.dumps(metadata, indent=2))
    # The existing private dataset has already been retrieved successfully in Cell 3.
    result = run_kaggle([
        'datasets', 'version', '-p', str(stage), '-m', message, '-t'
    ], timeout=900)
    if result.returncode:
        raise RuntimeError('Remote upload failed; STOP before the next run.\n' + (result.stderr + result.stdout)[-1500:])
    last_error = None
    for attempt in range(4):
        try:
            verify_uploaded_snapshot(rows)
            print('PRIVATE BACKUP SUCCESSFUL:', DATASET_REF)
            return rows
        except Exception as exc:
            last_error = exc
            if attempt < 3:
                time.sleep(15)
    raise RuntimeError('Remote backup could not be verified; STOP training: ' + str(last_error))

# Existing private backup already exists and contains at least 62 experiment records.
import_latest_backup()
assert len(load_metrics()) >= 62
assert len(completed_artifact_runs()) >= 40
print('BACKUP RESTORED AND READY:', len(load_metrics()), 'recorded;', len(pending()), 'missing')
print('The next cell trains and persists new experiments.')


In [ ]:
# CELL 4 / 5 — Train missing experiments and VERIFY a private backup after every run
# Rerun this cell to process another batch. Do not rerun any old Step 3 cells.

import os, sys, time, subprocess, hashlib
from IPython.display import FileLink, display
import torch

MAX_NEW_RUNS = 5          # Start with 5; increase to 8 or 10 only when stable.
MAX_SESSION_HOURS = 3.0  # Stop BETWEEN runs if this budget is reached.
RETRAIN_LOG_ONLY_FOR_DETAILS = False  # Keep False for the 58 not-yet-recorded runs.

assert torch.cuda.is_available(), 'Enable Kaggle T4 GPU.'
assert hashlib.sha256(TRAINER.read_bytes()).hexdigest() == EXPECTED_HASH
assert len(load_metrics()) >= 62, 'Restore the private backup before training.'

start = time.monotonic()
new_verified = 0
print('Starting from:', len(load_metrics()), 'records; missing:', len(pending(False)))
print('Next run:', pending(False)[0][3] if pending(False) else 'NONE')

while new_verified < MAX_NEW_RUNS:
    todo = pending(RETRAIN_LOG_ONLY_FOR_DETAILS)
    if not todo:
        print('ALL REQUESTED EXPERIMENTS COMPLETED.')
        break
    if (time.monotonic() - start) / 3600 >= MAX_SESSION_HOURS:
        print('Session time budget reached; stopped between experiments.')
        break
    fold, seed, variant, run_name = todo[0]
    print('\n' + '=' * 65, flush=True)
    print('TRAINING', run_name, '| missing:', len(todo), flush=True)
    print('=' * 65, flush=True)
    command = [
        sys.executable, '-u', str(GRID_SCRIPT),
        '--train-tsv', str(TRAIN), '--test-tsv', str(TEST),
        '--fold-assignment', str(FOLDS), '--output', str(GRID),
        '--folds', str(fold), '--seeds', str(seed), '--variants', variant,
        '--epochs', '2', '--batch-size', '24',
        '--checkpoint-every-steps', '250', '--max-runs', '1',
    ]
    finished = subprocess.run(command, cwd=str(ROOT), env=os.environ.copy())
    if finished.returncode != 0 or run_name not in completed_artifact_runs():
        print('Experiment incomplete; not marked as successful. STOPPING.')
        print('Local checkpoint (if any):', GRID / run_name / 'checkpoints/latest.pt')
        break
    print('Training completed. Uploading + VERIFYING remote backup...')
    # Fail closed: next experiment will not launch until a fresh backup is verified.
    try:
        count = push_private_backup(f'Completed {run_name}')
    except Exception as exc:
        print('BACKUP FAILED: STOPPING. The local result remains under:', GRID / run_name)
        print(str(exc))
        break
    new_verified += 1
    print('SAVED', count, '/120 metric records. Missing:', len(pending(False)), flush=True)

print('\nSESSION SUMMARY')
print('Metric records:', len(load_metrics()), '/120')
print('Genuine detailed results:', len(completed_artifact_runs()))
print('Log-only historical metrics:', len(set(load_metrics().experiment) - completed_artifact_runs()))
print('Remaining missing:', len(pending(False)))
print('New runs verified on Kaggle this batch:', new_verified)
print('Private dataset:', DATASET_REF)
if BACKUP_ZIP.exists(): display(FileLink(str(BACKUP_ZIP)))
if METRICS_PATH.exists(): display(FileLink(str(METRICS_PATH)))


In [ ]:
# CELL 5 / 5 — Optional read-only report (no GPU training)
from IPython.display import display, FileLink
print('Recorded metric rows:', len(load_metrics()), '/120')
print('Detailed run folders:', len(completed_artifact_runs()))
print('Historical log-only rows:', len(set(load_metrics().experiment) - completed_artifact_runs()))
print('Still missing:', len(pending(False)))
print('Kaggle private dataset:', DATASET_REF)
print('First five missing runs:', [x[3] for x in pending(False)[:5]])
display(load_metrics().tail(10))
if METRICS_PATH.is_file(): display(FileLink(str(METRICS_PATH)))
if BACKUP_ZIP.is_file(): display(FileLink(str(BACKUP_ZIP)))


In [ ]:

# ============================================================
# IJIES ROUND 2 — FINAL EXPERIMENT ANALYSIS
# Reviewer R1.5, R1.6, R2.4 and provenance checks
# ============================================================

import os, sys, subprocess, itertools, json, hashlib
import shutil, platform
from pathlib import Path

import numpy as np
import pandas as pd

REF = "gomathybalaji/ijies-vircon62-resume-private"
OUT = Path("/kaggle/working/IJIES_Round2_Analysis")
OUT.mkdir(parents=True, exist_ok=True)

FILENAME = "IJIES_Recovery_Combined_Metrics.csv"

# ------------------------------------------------------------
# 1. FIND THE MOST COMPLETE METRICS FILE
# ------------------------------------------------------------

candidates = []

if "METRICS_PATH" in globals():
    p = Path(METRICS_PATH)
    if p.is_file():
        candidates.append(p)

candidates += list(
    Path("/kaggle/input").rglob(FILENAME)
)

candidates += list(
    Path("/kaggle/working").rglob(FILENAME)
)

def valid_candidate(p):
    try:
        return len(pd.read_csv(p))
    except Exception:
        return -1

best = max(candidates, key=valid_candidate) if candidates else None

# Download the current private dataset if a complete
# local copy is not already available.
if best is None or valid_candidate(best) < 120:

    from kaggle_secrets import UserSecretsClient

    client = UserSecretsClient()

    os.environ["KAGGLE_USERNAME"] = client.get_secret(
        "KAGGLE_USERNAME"
    )

    os.environ["KAGGLE_KEY"] = client.get_secret(
        "KAGGLE_KEY"
    )

    download_dir = OUT / "remote_backup"
    download_dir.mkdir(exist_ok=True)

    result = subprocess.run(
        [
            "kaggle", "datasets", "download", REF,
            "-p", str(download_dir),
            "--unzip", "-o", "-q"
        ],
        capture_output=True,
        text=True,
        timeout=600
    )

    if result.returncode != 0:
        raise RuntimeError(
            "Could not download the verified backup:\n"
            + result.stderr[-1000:]
        )

    candidates = list(
        download_dir.rglob(FILENAME)
    )

    if not candidates:
        raise FileNotFoundError(
            "Metrics CSV not found in downloaded backup."
        )

    best = max(candidates, key=valid_candidate)

metrics = pd.read_csv(best)

required = {
    "experiment", "fold", "seed", "variant",
    "macro_f1", "accuracy", "balanced_accuracy"
}

assert required.issubset(metrics.columns)
assert metrics["experiment"].is_unique

for c in ["macro_f1", "accuracy", "balanced_accuracy"]:
    metrics[c] = pd.to_numeric(metrics[c], errors="raise")
    assert metrics[c].between(0, 1).all()

metrics["fold"] = metrics["fold"].astype(int)
metrics["seed"] = metrics["seed"].astype(int)

SEEDS = [20266008, 20266009, 20266010]
FOLDS = [1, 2, 3, 4, 5]

VARIANTS = [
    "full", "no_event", "no_contrastive",
    "no_request", "no_triage",
    "distilbert", "crisis", "tfidf_svc"
]

expected = set(itertools.product(
    FOLDS, SEEDS, VARIANTS
))

observed = set(zip(
    metrics.fold,
    metrics.seed,
    metrics.variant
))

assert observed == expected, (
    f"Missing: {len(expected - observed)}; "
    f"Unexpected: {len(observed - expected)}"
)

print("VERIFIED: 120/120 unique experiments")

# ------------------------------------------------------------
# 2. OVERALL MODEL COMPARISON
# ------------------------------------------------------------

summary = (
    metrics.groupby("variant")
    .agg(
        runs=("macro_f1", "size"),
        mean_macro_f1=("macro_f1", "mean"),
        sd_macro_f1=("macro_f1", "std"),
        mean_accuracy=("accuracy", "mean"),
        mean_balanced_accuracy=(
            "balanced_accuracy", "mean"
        )
    )
    .sort_values("mean_macro_f1", ascending=False)
    .reset_index()
)

summary.to_csv(
    OUT / "Table_Model_Comparison.csv",
    index=False
)

print("\nMODEL COMPARISON")
print(summary.round(4).to_string(index=False))

# ------------------------------------------------------------
# 3. MATCHED COMPONENT ABLATIONS
# ------------------------------------------------------------

wide = metrics.pivot(
    index=["fold", "seed"],
    columns="variant",
    values="macro_f1"
)

assert wide.shape == (15, 8)
assert not wide.isna().any().any()

rng = np.random.default_rng(20261121)
comparisons = []

for variant in VARIANTS:

    if variant == "full":
        continue

    # Positive delta means full model performed better.
    delta = wide["full"] - wide[variant]

    # Seed-average within each fold before inference.
    fold_delta = delta.groupby(level="fold").mean()

    values = fold_delta.to_numpy()

    # Resample the five outer folds rather than
    # incorrectly treating all 15 runs as independent.
    bootstrap = rng.choice(
        values,
        size=(10000, len(values)),
        replace=True
    ).mean(axis=1)

    ci_low, ci_high = np.quantile(
        bootstrap, [0.025, 0.975]
    )

    observed_delta = float(values.mean())

    # Exact two-sided sign-flip test over five folds.
    null = []

    for signs in itertools.product([-1, 1], repeat=5):
        null.append(
            float(np.mean(values * np.array(signs)))
        )

    p_value = float(
        np.mean(
            np.abs(null) >= abs(observed_delta) - 1e-12
        )
    )

    comparisons.append({
        "comparison": f"full_minus_{variant}",
        "paired_runs": 15,
        "independent_resampling_units": 5,
        "mean_delta_macro_f1": observed_delta,
        "ci95_low": float(ci_low),
        "ci95_high": float(ci_high),
        "exact_signflip_p": p_value,
        "folds_favoring_full": int((values > 0).sum()),
        "folds_favoring_comparator": int((values < 0).sum())
    })

paired = pd.DataFrame(comparisons)

# Holm multiple-comparison adjustment.
pvals = paired["exact_signflip_p"].to_numpy()
order = np.argsort(pvals)
adjusted = np.empty(len(pvals))
previous = 0.0

for rank, idx in enumerate(order):
    previous = max(
        previous, (len(pvals) - rank) * pvals[idx]
    )
    adjusted[idx] = min(1.0, previous)

paired["holm_adjusted_p"] = adjusted

paired.to_csv(
    OUT / "Table_Paired_Ablation_Analysis.csv",
    index=False
)

print("\nMATCHED PAIRED ANALYSIS")
print(paired.round(4).to_string(index=False))

# ------------------------------------------------------------
# 4. FOLD-WISE AND SEED-WISE PERFORMANCE
# ------------------------------------------------------------

fold_summary = (
    metrics.groupby(["fold", "variant"])
    .agg(
        mean_macro_f1=("macro_f1", "mean"),
        sd_across_seeds=("macro_f1", "std")
    )
    .reset_index()
)

fold_summary.to_csv(
    OUT / "Table_Per_Fold_Seed_Variation.csv",
    index=False
)

# ------------------------------------------------------------
# 5. AUDIT SAVED RUN ARTIFACTS
# ------------------------------------------------------------

grid_candidates = [best.parent / "grid"]

if "GRID" in globals():
    grid_candidates.append(Path(GRID))

grid_candidates = [
    p for p in grid_candidates if p.is_dir()
]

if not grid_candidates:
    raise RuntimeError(
        "Metrics are complete, but detailed artifact grid "
        "is unavailable. Attach or restore the backup."
    )

grid = max(
    grid_candidates,
    key=lambda p: len(list(p.rglob("SUCCESS.json")))
)

inventory = []

for experiment in metrics["experiment"]:

    folder = grid / experiment

    inventory.append({
        "experiment": experiment,
        "has_success": (
            folder / "SUCCESS.json"
        ).is_file(),
        "has_summary": (
            folder / "summary.json"
        ).is_file(),
        "has_per_event": (
            folder / "per_event.csv"
        ).is_file(),
        "has_per_class": (
            folder / "per_class.csv"
        ).is_file(),
        "has_predictions": (
            folder / "predictions_no_text.csv"
        ).is_file(),
        "has_training_history": (
            folder / "epoch_history.csv"
        ).is_file(),
        "has_run_manifest": (
            folder / "run_manifest.json"
        ).is_file(),
        "has_checkpoint": any(
            folder.rglob("*.pt")
        ) or any(folder.rglob("*.safetensors"))
    })

inventory = pd.DataFrame(inventory)

inventory.to_csv(
    OUT / "Table_Artifact_Completeness.csv",
    index=False
)

print("\nARTIFACT COMPLETENESS")
print(inventory.drop(columns="experiment").sum())

# ------------------------------------------------------------
# 6. COMBINE THE AVAILABLE PER-EVENT FILES
# ------------------------------------------------------------

per_event_tables = []

for experiment in metrics["experiment"]:

    path = grid / experiment / "per_event.csv"

    if not path.exists():
        continue

    table = pd.read_csv(path)

    table.insert(0, "experiment", experiment)

    per_event_tables.append(table)

if per_event_tables:

    combined = pd.concat(
        per_event_tables,
        ignore_index=True
    )

    combined.to_csv(
        OUT / "Supplementary_Per_Event_Results.csv",
        index=False
    )

    print(
        "\nPer-event files combined:",
        len(per_event_tables)
    )

# ------------------------------------------------------------
# 7. GENERATE COMPARISON FIGURE
# ------------------------------------------------------------

import matplotlib.pyplot as plt

plot_data = summary.sort_values(
    "mean_macro_f1"
)

plt.figure(figsize=(10, 5.5))

plt.barh(
    plot_data["variant"],
    plot_data["mean_macro_f1"],
    xerr=plot_data["sd_macro_f1"],
    capsize=3
)

plt.xlabel("Mean macro-F1")
plt.ylabel("Model variant")
plt.title(
    "Exploratory event-disjoint evaluation: "
    "5 folds × 3 seeds"
)

plt.tight_layout()

plt.savefig(
    OUT / "Figure_Model_Comparison.png",
    dpi=300
)

plt.show()

# ------------------------------------------------------------
# 8. ENVIRONMENT AND REPORT
# ------------------------------------------------------------

manifest = {
    "analysis_type": "exploratory_development_event_folds",
    "experiment_rows": len(metrics),
    "folds": FOLDS,
    "seeds": SEEDS,
    "variants": VARIANTS,
    "python_version": platform.python_version(),
    "metrics_file_sha256": hashlib.sha256(
        best.read_bytes()
    ).hexdigest(),
    "important_limitation": (
        "These folds informed earlier model development. "
        "They are not independent untouched-event confirmation."
    )
}

(OUT / "Analysis_Manifest.json").write_text(
    json.dumps(manifest, indent=2)
)

# Zip all generated tables and figure.
archive = shutil.make_archive(
    str(OUT),
    "zip",
    root_dir=str(OUT)
)

print("\n" + "=" * 65)
print("ROUND 2 STATISTICAL ANALYSIS COMPLETE")
print("=" * 65)
print("Results folder:", OUT)
print("Results ZIP:", archive)


In [ ]:

# ============================================================
# IJIES ROUND 2 — INDEPENDENT EVENT DATASET PREPARATION
# CORRECTED KAGGLE VERSION
# ============================================================

from pathlib import Path
import pandas as pd
import hashlib
import json
import re

# ------------------------------------------------------------
# 1. AUTOMATICALLY LOCATE ATTACHED DATASET
# ------------------------------------------------------------

INPUT_ROOT = Path("/kaggle/input")

candidates = list(INPUT_ROOT.rglob("tweets.csv"))

# Prefer the Turkey/Syria Earthquake dataset.
matches = [
    p for p in candidates
    if "turkey" in str(p).lower()
    or "syria" in str(p).lower()
]

if len(matches) == 1:
    source_file = matches[0]
elif len(matches) > 1:
    raise RuntimeError(
        "Multiple Turkey/Syria tweets.csv files found:\n"
        + "\n".join(map(str, matches))
    )
elif len(candidates) == 1:
    source_file = candidates[0]
else:
    raise FileNotFoundError(
        "Cannot uniquely locate tweets.csv.\n"
        "Found:\n" + "\n".join(map(str, candidates))
    )

print("DATASET FOUND SUCCESSFULLY")
print("File:", source_file)

# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------

df = pd.read_csv(
    source_file,
    low_memory=False
)

print("\nOriginal rows:", len(df))
print("Columns:", df.columns.tolist())

# ------------------------------------------------------------
# 3. IDENTIFY TEXT AND LANGUAGE COLUMNS
# ------------------------------------------------------------

columns = {
    str(c).strip().lower(): c
    for c in df.columns
}

def choose_column(names):
    for name in names:
        if name in columns:
            return columns[name]
    return None

text_col = choose_column([
    "text", "tweet", "tweet_text",
    "full_text", "content",
    "rawcontent", "raw_content"
])

lang_col = choose_column([
    "lang", "language", "tweet_language"
])

if text_col is None:
    raise RuntimeError(
        "Text column not found. Available columns: "
        + str(df.columns.tolist())
    )

if lang_col is None:
    raise RuntimeError(
        "Language column not found. Available columns: "
        + str(df.columns.tolist())
    )

# ------------------------------------------------------------
# 4. FILTER ENGLISH TWEETS
# ------------------------------------------------------------

english = df[
    df[lang_col]
    .astype(str)
    .str.lower()
    .str.strip()
    .isin(["en", "english"])
].copy()

english["text"] = (
    english[text_col]
    .fillna("")
    .astype(str)
    .str.strip()
)

english = english[
    english["text"].str.len() >= 15
].copy()

# ------------------------------------------------------------
# 5. REMOVE EXACT TEXT DUPLICATES
# ------------------------------------------------------------

def normalized_hash(text):
    normalized = re.sub(
        r"\s+", " ", str(text).casefold().strip()
    )
    return hashlib.sha256(
        normalized.encode("utf-8")
    ).hexdigest()

english["text_hash"] = (
    english["text"].map(normalized_hash)
)

english = english.drop_duplicates(
    subset=["text_hash"]
).copy()

print("Unique English messages:", len(english))

if len(english) == 0:
    raise RuntimeError(
        "No usable English-language messages found."
    )

# ------------------------------------------------------------
# 6. SELECT REPRODUCIBLE SAMPLE
# ------------------------------------------------------------

SAMPLE_SIZE = 600
SEED = 20261121

sample = english.sample(
    n=min(SAMPLE_SIZE, len(english)),
    random_state=SEED
).copy()

sample = sample.reset_index(drop=True)

sample["message_id"] = [
    "TR2023_" + value[:20]
    for value in sample["text_hash"]
]

sample["event_id"] = (
    "turkey_syria_earthquake_2023"
)

result = sample[
    ["message_id", "event_id", "text"]
].copy()

# ------------------------------------------------------------
# 7. SAVE DATASET
# ------------------------------------------------------------

OUT = Path(
    "/kaggle/working/IJIES_Round2_Analysis"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

result.to_csv(
    OUT / "new_events.csv",
    index=False
)

# ------------------------------------------------------------
# 8. CREATE HUMAN ANNOTATION TEMPLATES
# ------------------------------------------------------------

for rater in [1, 2]:

    annotation = result.copy()

    annotation["humanitarian_label_11"] = ""
    annotation["actionability_label"] = ""

    annotation.to_csv(
        OUT / f"Independent_Rater_{rater}.csv",
        index=False
    )

# ------------------------------------------------------------
# 9. SAVE EXPERIMENT MANIFEST
# ------------------------------------------------------------

manifest = {
    "source_file": str(source_file),
    "source_event": "Turkey Syria Earthquake 2023",
    "language": "English",
    "sample_size": len(result),
    "random_seed": SEED,
    "unique_events": 1,
    "human_labels": "Pending",
    "independence": "Not yet fully verified"
}

(OUT / "New_Event_Manifest.json").write_text(
    json.dumps(manifest, indent=2)
)

# ------------------------------------------------------------
# 10. DISPLAY FINAL OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("DATASET PREPARATION COMPLETED")
print("=" * 65)

print("Original tweets:", len(df))
print("Usable unique English tweets:", len(english))
print("Selected messages:", len(result))
print("New disaster events:", result.event_id.nunique())

print("\nFiles created:")
print(OUT / "new_events.csv")
print(OUT / "Independent_Rater_1.csv")
print(OUT / "Independent_Rater_2.csv")
print(OUT / "New_Event_Manifest.json")

print("\nPREVIEW:")
print(result.head(3).to_string(index=False))

print("\nNEXT: Verify event and text overlap.")
print("Humanitarian and actionability labels remain pending.")


In [ ]:

# ============================================================
# IJIES ROUND 2 — NEW EVENT / ANNOTATION PREPARATION
# R1.1, R1.7, R2.1, R2.2
# ============================================================

import pandas as pd
import numpy as np
import hashlib
import json
import re
from pathlib import Path

OUT = Path("/kaggle/working/IJIES_Round2_Analysis")
OUT.mkdir(parents=True, exist_ok=True)

# Attach a separate Kaggle dataset containing new events.
# Update this path to the actual CSV you upload.
NEW_EVENTS_FILE = Path(
    "/kaggle/input/vircon-new-events/new_events.csv"
)

# Create the required schema if new data is not attached.
if not NEW_EVENTS_FILE.exists():

    pd.DataFrame(columns=[
        "message_id",
        "event_id",
        "text"
    ]).to_csv(
        OUT / "REQUIRED_New_Events_Template.csv",
        index=False
    )

    print("NEW EXTERNAL EVENT DATASET REQUIRED")
    print("Template created.")
    print("Do not treat historical events as untouched.")

else:

    new_data = pd.read_csv(NEW_EVENTS_FILE)

    required = {"message_id", "event_id", "text"}

    assert required.issubset(new_data.columns), (
        "New-event CSV needs message_id, event_id, text."
    )

    assert new_data["message_id"].notna().all()
    assert new_data["event_id"].notna().all()
    assert new_data["text"].notna().all()
    assert new_data["message_id"].is_unique

    # Original fold assignment must be available.
    if "FOLDS" in globals() and Path(FOLDS).is_file():
        assignment_file = Path(FOLDS)
    else:
        options = list(
            Path("/kaggle/input").rglob(
                "event_disjoint_fold_assignment.csv"
            )
        )
        assert options, (
            "Original canonical event assignment is required "
            "to verify event disjointness."
        )
        assignment_file = options[0]

    previous = pd.read_csv(assignment_file)

    event_columns = [
        "canonical_event",
        "canonical_event_id",
        "event_id",
        "event_name",
        "disaster_event",
        "event"
    ]

    event_col = next(
        (c for c in event_columns if c in previous.columns),
        None
    )

    if event_col is None:
        raise ValueError(
            "Cannot identify canonical event column. "
            f"Available columns: {previous.columns.tolist()}"
        )

    old_events = set(
        previous[event_col]
        .dropna()
        .astype(str)
        .str.strip()
        .str.lower()
    )

    new_events = set(
        new_data["event_id"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    event_overlap = old_events & new_events

    print("Original events:", len(old_events))
    print("New events:", len(new_events))
    print("Matching event IDs:", len(event_overlap))

    if event_overlap:
        raise RuntimeError(
            "Event leakage detected: "
            + str(sorted(event_overlap)[:10])
        )

    # Exact duplicate-text screening.
    # Near duplicates still require additional review.
    def text_hash(x):
        x = re.sub(
            r"\s+", " ", str(x).lower().strip()
        )
        return hashlib.sha256(
            x.encode("utf-8")
        ).hexdigest()

    old_texts = set()

    original_names = [
        "crisis_consolidated_humanitarian_filtered_lang_en_train.tsv",
        "crisis_consolidated_humanitarian_filtered_lang_en_test.tsv"
    ]

    for filename in original_names:

        matches = list(
            Path("/kaggle/input").rglob(filename)
        )

        assert matches, (
            "Original train/test data must be attached "
            "for duplicate-text verification: " + filename
        )

        old_df = pd.read_csv(matches[0], sep="\t")

        text_col = next(
            (
                c for c in [
                    "text", "tweet_text", "tweet",
                    "message", "message_text"
                ]
                if c in old_df.columns
            ),
            None
        )

        if text_col is None:
            raise ValueError(
                f"Identify text column in {filename}: "
                f"{old_df.columns.tolist()}"
            )

        old_texts.update(
            old_df[text_col].dropna().map(text_hash)
        )

    new_data["text_sha256"] = (
        new_data["text"].map(text_hash)
    )

    duplicate_count = int(
        new_data["text_sha256"].isin(old_texts).sum()
    )

    print("Exact text overlaps:", duplicate_count)

    if duplicate_count:
        raise RuntimeError(
            "New events contain previously used text. "
            "Review duplicates before proceeding."
        )

    # Create an event-stratified, blinded annotation sample.
    # The sample is not automatically considered sufficient
    # for rare request/response categories.
    rng = np.random.default_rng(20261121)

    sampled = []

    for event_id, group in new_data.groupby("event_id"):

        n = min(30, len(group))

        chosen = group.sample(
            n=n,
            random_state=int(
                rng.integers(1, 2**31 - 1)
            )
        )

        sampled.append(chosen)

    sample = pd.concat(
        sampled,
        ignore_index=True
    )

    sample = sample[
        ["message_id", "event_id", "text"]
    ].copy()

    sample["human_label"] = ""

    sample.to_csv(
        OUT / "Rater_1_Blinded.csv",
        index=False
    )

    sample.to_csv(
        OUT / "Rater_2_Blinded.csv",
        index=False
    )

    adjudication = sample.drop(
        columns="human_label"
    ).copy()

    adjudication["rater_1"] = ""
    adjudication["rater_2"] = ""
    adjudication["adjudicated_label"] = ""

    adjudication.to_csv(
        OUT / "Adjudication_Template.csv",
        index=False
    )

    protocol = {
        "allowed_labels": [
            "request",
            "response",
            "other",
            "uncertain"
        ],
        "sampling_seed": 20261121,
        "sampling_method": "event-stratified random sample",
        "messages_sampled": len(sample),
        "events_sampled": sample.event_id.nunique(),
        "new_source_sha256": hashlib.sha256(
            NEW_EVENTS_FILE.read_bytes()
        ).hexdigest(),
        "status": "Awaiting independent human annotations",
        "note": (
            "Annotators must not view model predictions. "
            "Resolve disagreements with documented adjudication. "
            "Additional targeted sampling may be necessary "
            "to obtain enough rare requests and responses."
        )
    }

    (OUT / "Annotation_Protocol.json").write_text(
        json.dumps(protocol, indent=2)
    )

    print("\nNEW EVENT SCREENING PASSED")
    print("Annotation rows:", len(sample))
    print("Independent events:", sample.event_id.nunique())
    print("Rater templates created.")
    print("Human annotation is still required.")


In [ ]:

# ============================================================
# IJIES ROUND 2 — STEP 2
# CHECK DATA INDEPENDENCE + PREPARE HUMAN ANNOTATION
# No GPU required
# ============================================================

from pathlib import Path
import pandas as pd
import hashlib
import json
import re
import os

OUT = Path("/kaggle/working/IJIES_Round2_Analysis")
NEW_FILE = OUT / "new_events.csv"

assert NEW_FILE.exists(), (
    "Run the Turkey-Syria dataset preparation cell first."
)

new = pd.read_csv(NEW_FILE)

assert {"message_id", "event_id", "text"}.issubset(
    new.columns
)
assert len(new) == 600
assert new.message_id.is_unique
assert new.text.notna().all()

print("=" * 65)
print("IJIES INDEPENDENT DATASET VERIFICATION")
print("=" * 65)

print("New dataset records:", len(new))
print("New events:", new.event_id.nunique())

# ------------------------------------------------------------
# 1. IDENTIFY ORIGINAL FOLD ASSIGNMENT
# ------------------------------------------------------------

def find_input_file(filename, global_name=None):

    if global_name and global_name in globals():
        value = globals()[global_name]

        if isinstance(value, (str, os.PathLike)):
            path = Path(value)
            if path.is_file():
                return path

    for root in [
        Path("/kaggle/input"),
        Path("/kaggle/working")
    ]:
        matches = sorted(root.rglob(filename))

        if matches:
            return matches[0]

    return None

fold_file = find_input_file(
    "event_disjoint_fold_assignment.csv",
    "FOLDS"
)

event_overlap = None
fold_event_column = None

if fold_file is not None:

    old_folds = pd.read_csv(fold_file)

    candidate_columns = [
        "canonical_event",
        "canonical_event_name",
        "canonical_event_id",
        "event_id",
        "event_name",
        "event"
    ]

    fold_event_column = next(
        (
            c for c in candidate_columns
            if c in old_folds.columns
        ),
        None
    )

    if fold_event_column:

        def normalize_event(value):
            return re.sub(
                r"[^a-z0-9]+",
                "_",
                str(value).casefold()
            ).strip("_")

        original_events = set(
            old_folds[fold_event_column]
            .dropna()
            .map(normalize_event)
        )

        new_events = set(
            new.event_id.map(normalize_event)
        )

        event_overlap = sorted(
            original_events & new_events
        )

        print("\nORIGINAL EVENT CHECK")
        print("Assignment:", fold_file)
        print("Event column:", fold_event_column)
        print("Original event identifiers:", len(original_events))
        print("Exact normalized event overlap:", len(event_overlap))

        if event_overlap:
            print("OVERLAPPING EVENTS:", event_overlap)
        else:
            print("No exact event identifier overlap found.")
            print("Event aliases/provenance still require review.")

    else:
        print("\nCannot identify event column.")
        print("Columns:", old_folds.columns.tolist())

else:
    print("\nOriginal event assignment not located.")
    print("Event-independence screening is incomplete.")

# ------------------------------------------------------------
# 2. CHECK EXACT TEXT OVERLAP WITH ORIGINAL CRISIS DATA
# ------------------------------------------------------------

def normalize_text_hash(value):

    normalized = re.sub(
        r"\s+", " ",
        str(value).casefold().strip()
    )

    return hashlib.sha256(
        normalized.encode("utf-8")
    ).hexdigest()

old_names = [
    (
        "crisis_consolidated_humanitarian_"
        "filtered_lang_en_train.tsv"
    ),
    (
        "crisis_consolidated_humanitarian_"
        "filtered_lang_en_test.tsv"
    )
]

old_text_hashes = set()
checked_files = []
missing_files = []

for filename in old_names:

    path = find_input_file(filename)

    if path is None:
        missing_files.append(filename)
        continue

    old = pd.read_csv(
        path, sep="\t", low_memory=False
    )

    possible_text_columns = [
        "text", "tweet_text", "tweet",
        "message", "message_text",
        "clean_text", "full_text", "content"
    ]

    text_col = next(
        (
            col for col in possible_text_columns
            if col in old.columns
        ),
        None
    )

    if text_col is None:
        print("Cannot find text column in:", filename)
        print("Available:", old.columns.tolist())
        missing_files.append(filename + " (text column unresolved)")
        continue

    old_text_hashes.update(
        old[text_col]
        .dropna()
        .map(normalize_text_hash)
    )

    checked_files.append(str(path))

new["text_hash"] = new.text.map(
    normalize_text_hash
)

overlap_count = None

if len(checked_files) == 2:

    overlap_count = int(
        new.text_hash.isin(old_text_hashes).sum()
    )

    print("\nTEXT DUPLICATE CHECK")
    print("Original datasets checked:", len(checked_files))
    print("Original unique text hashes:", len(old_text_hashes))
    print("Overlapping new messages:", overlap_count)

else:
    print("\nText-overlap check incomplete.")
    print("Missing inputs:", missing_files)

# ------------------------------------------------------------
# 3. CREATE BLINDED HUMAN ANNOTATION SHEETS
# ------------------------------------------------------------

# Original paper's eleven humanitarian labels.
HUMANITARIAN_CLASSES = [
    "affected_individual",
    "caution_and_advice",
    "displaced_and_evacuations",
    "donation_and_volunteering",
    "infrastructure_and_utilities_damage",
    "injured_or_dead_people",
    "missing_and_found_people",
    "not_humanitarian",
    "requests_or_needs",
    "response_efforts",
    "sympathy_and_support"
]

ACTION_CLASSES = [
    "request",
    "response",
    "other",
    "uncertain"
]

annotation = new[
    ["message_id", "event_id", "text"]
].copy()

annotation["disaster_relevance"] = ""
annotation["humanitarian_label_11"] = ""
annotation["actionability_label"] = ""
annotation["annotation_notes"] = ""

# Preserve the original sample and identifiers.
for rater, seed in [(1, 101), (2, 202)]:

    blinded = annotation.sample(
        frac=1,
        random_state=seed
    ).reset_index(drop=True)

    blinded.to_csv(
        OUT / f"Independent_Rater_{rater}.csv",
        index=False
    )

guide = {
    "humanitarian_labels": HUMANITARIAN_CLASSES,
    "actionability_labels": ACTION_CLASSES,
    "disaster_relevance_values": [
        "relevant",
        "not_relevant",
        "uncertain"
    ],
    "instructions": [
        "Annotate independently without seeing model predictions.",
        "Apply the paper's original eleven-class label definitions.",
        "Use request/response/other operational definitions.",
        "Mark off-topic messages as not_relevant.",
        "Use uncertain where the evidence is insufficient.",
        "Preserve all rows, including irrelevant messages.",
        "Adjudicate disagreements after both raters finish."
    ]
}

(OUT / "Human_Annotation_Guide.json").write_text(
    json.dumps(guide, indent=2)
)

# ------------------------------------------------------------
# 4. SAVE SCREENING REPORT
# ------------------------------------------------------------

if event_overlap is None or overlap_count is None:
    status = "INCOMPLETE_CHECKS"
elif event_overlap or overlap_count:
    status = "POTENTIAL_OVERLAP_REQUIRES_REVIEW"
else:
    status = "NO_EXACT_OVERLAP_FOUND_MANUAL_REVIEW_REQUIRED"

report = {
    "new_dataset_rows": len(new),
    "new_event_count": int(new.event_id.nunique()),
    "original_fold_file": (
        str(fold_file) if fold_file else None
    ),
    "original_event_column": fold_event_column,
    "event_overlap": event_overlap,
    "original_text_files_checked": checked_files,
    "missing_text_files": missing_files,
    "exact_text_overlap_count": overlap_count,
    "screening_status": status,
    "additional_verification_required": [
        "Review event aliases and dates",
        "Check near duplicates and reposts",
        "Confirm events were never used during development",
        "Obtain independent human labels",
        "Freeze model and threshold before final evaluation"
    ]
}

(OUT / "Independence_Check_Report.json").write_text(
    json.dumps(report, indent=2)
)

print("\n" + "=" * 65)
print("INDEPENDENCE SCREENING")
print("=" * 65)
print("Status:", status)
print("Event identifier overlap:", event_overlap)
print("Exact message overlap:", overlap_count)

print("\nANNOTATION FILES")
print(OUT / "Independent_Rater_1.csv")
print(OUT / "Independent_Rater_2.csv")
print(OUT / "Human_Annotation_Guide.json")
print(OUT / "Independence_Check_Report.json")

print("\nNext: complete independent human annotations.")
print("Do not treat this sample as a labelled final test yet.")


In [ ]:

from pathlib import Path
import zipfile
import pandas as pd

print("=" * 60)
print("FIND ORIGINAL EVENT ASSIGNMENT")
print("=" * 60)

found = []

for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
    for p in root.rglob("event_disjoint_fold_assignment.csv"):
        found.append(p)

if found:
    for p in found:
        print("\nFOUND:", p)
        df = pd.read_csv(p)
        print("Rows:", len(df))
        print("Columns:", df.columns.tolist())

else:
    print("Original fold assignment not found as a CSV.")
    print("Checking source archives...")

    for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
        for archive_path in root.rglob("*.zip"):
            try:
                with zipfile.ZipFile(archive_path) as z:
                    matches = [
                        n for n in z.namelist()
                        if n.endswith(
                            "event_disjoint_fold_assignment.csv"
                        )
                    ]
                    for member in matches:
                        destination = Path(
                            "/kaggle/working/original_event_assignment"
                        )
                        destination.mkdir(
                            parents=True, exist_ok=True
                        )
                        data = z.read(member)
                        output = destination / (
                            "event_disjoint_fold_assignment.csv"
                        )
                        if output.exists() and output.read_bytes() != data:
                            raise RuntimeError(
                                "Conflicting event assignment archives"
                            )
                        output.write_bytes(data)
                        found.append(output)
                        print("RECOVERED:", output)
            except (zipfile.BadZipFile, OSError):
                continue

if not found:
    print("\nEVENT ASSIGNMENT STILL MISSING")
    print("Attach the original experiment source ZIP")
    print("or the fold assignment CSV used for 120 runs.")
else:
    print("\nORIGINAL EVENT ASSIGNMENT AVAILABLE")
    print("Use this file in the previous overlap-check cell.")


In [ ]:

# VERIFY NEW-EVENT NAME AGAINST ORIGINAL ASSIGNMENT

from pathlib import Path
import pandas as pd
import re

matches = list(
    Path("/kaggle/input").rglob(
        "event_disjoint_fold_assignment.csv"
    )
)

if not matches:
    raise FileNotFoundError(
        "Attach the recovered event assignment CSV first."
    )

original = pd.read_csv(matches[0])

event_columns = [
    c for c in original.columns
    if "event" in c.lower()
]

print("Original assignment:", matches[0])
print("Original rows:", len(original))
print("Available event columns:", event_columns)

if not event_columns:
    raise RuntimeError(
        "No event column identified. "
        "Inspect the printed columns."
    )

# Screen for aliases relating to the new disaster.
pattern = (
    r"turk|türki|syria|kahraman"
    r"|gaziantep|hatay|2023"
)

matches_by_row = original[event_columns].fillna("").astype(
    str
).apply(
    lambda col: col.str.contains(
        pattern, case=False, regex=True
    )
).any(axis=1)

overlaps = original.loc[matches_by_row]

print("\nPotential event-name matches:", len(overlaps))

if len(overlaps):
    print(
        overlaps[event_columns]
        .drop_duplicates()
        .head(30)
        .to_string(index=False)
    )
    print("\nMANUAL REVIEW REQUIRED.")
else:
    print("No matching event aliases found.")
    print("Exact text overlap previously checked: 0.")
    print(
        "Candidate new event passes this name screening; "
        "provenance and near-duplicate review remain."
    )


In [ ]:

# ============================================================
# IJIES ROUND 2 — NEAR-DUPLICATE SCREENING
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer

OUT = Path("/kaggle/working/IJIES_Round2_Analysis")
NEW_FILE = OUT / "new_events.csv"

DATA_DIR = Path(
    "/kaggle/input/datasets/gomathybalaji/crisis"
)

train_file = DATA_DIR / (
    "crisis_consolidated_humanitarian_filtered_lang_en_train.tsv"
)

test_file = DATA_DIR / (
    "crisis_consolidated_humanitarian_filtered_lang_en_test.tsv"
)

assert NEW_FILE.is_file(), "New-event CSV missing."
assert train_file.is_file(), "Original train TSV missing."
assert test_file.is_file(), "Original test TSV missing."

new = pd.read_csv(NEW_FILE)

old_train = pd.read_csv(
    train_file, sep="\t", low_memory=False
)
old_test = pd.read_csv(
    test_file, sep="\t", low_memory=False
)

def find_text_column(df):
    for column in [
        "text", "tweet_text", "tweet",
        "message", "message_text", "content"
    ]:
        if column in df.columns:
            return column

    raise ValueError(
        "Text column missing. Found: "
        + str(df.columns.tolist())
    )

train_text_col = find_text_column(old_train)
test_text_col = find_text_column(old_test)

old_text = pd.concat([
    old_train[train_text_col],
    old_test[test_text_col]
], ignore_index=True).fillna("").astype(str)

def normalize_text(text):

    text = str(text).casefold()

    text = re.sub(r"https?://\S+", " URL ", text)
    text = re.sub(r"@\w+", " USER ", text)
    text = re.sub(r"\brt\s+", "", text)

    text = re.sub(r"\W+", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

old_normalized = old_text.map(normalize_text)
new_normalized = new["text"].fillna("").map(normalize_text)

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=120000,
    sublinear_tf=True,
    norm="l2"
)

old_vectors = vectorizer.fit_transform(
    old_normalized
)

new_vectors = vectorizer.transform(
    new_normalized
)

best_scores = []
best_indices = []

for start in range(0, len(new), 50):

    stop = min(start + 50, len(new))

    similarity = (
        new_vectors[start:stop]
        @ old_vectors.T
    )

    scores = (
        similarity.max(axis=1)
        .toarray()
        .ravel()
    )

    indices = np.asarray(
        similarity.argmax(axis=1)
    ).ravel()

    best_scores.extend(scores)
    best_indices.extend(indices)

review = new.copy()

review["max_similarity"] = best_scores

review["closest_original_text"] = [
    old_text.iloc[int(i)]
    for i in best_indices
]

# Flag potential overlaps for human review.
# The threshold is a screening heuristic,
# not a proof of duplication.
review["manual_review_required"] = (
    review["max_similarity"] >= 0.80
)

review = review.sort_values(
    "max_similarity",
    ascending=False
)

review.to_csv(
    OUT / "Near_Duplicate_Review.csv",
    index=False
)

flagged = int(
    review["manual_review_required"].sum()
)

print("=" * 60)
print("NEAR-DUPLICATE SCREENING COMPLETED")
print("=" * 60)

print("New messages checked:", len(new))
print("Original messages checked:", len(old_text))
print("High-similarity messages:", flagged)

print("\nHighest similarity examples:")

print(
    review[
        ["message_id", "max_similarity"]
    ].head(10).to_string(index=False)
)

print("\nSaved:")
print(OUT / "Near_Duplicate_Review.csv")

print("\nReview flagged texts manually.")
print("Do not automatically remove messages.")


In [ ]:

# ============================================================
# IJIES ROUND 2 — NEAR-DUPLICATE MANUAL REVIEW LEDGER
# ============================================================

from pathlib import Path
import pandas as pd

OUT = Path("/kaggle/working/IJIES_Round2_Analysis")

df = pd.read_csv(
    OUT / "Near_Duplicate_Review.csv"
)

# Include borderline similarities for conservative review.
audit = df[
    df["max_similarity"] >= 0.75
].copy()

audit["review_decision"] = ""
audit["review_notes"] = ""

audit["review_decision_options"] = (
    "true_duplicate | same_template_different_facts "
    "| not_duplicate | uncertain"
)

audit = audit[
    [
        "message_id",
        "text",
        "closest_original_text",
        "max_similarity",
        "review_decision",
        "review_notes",
        "review_decision_options"
    ]
]

path = OUT / "Near_Duplicate_Manual_Review.csv"

audit.to_csv(path, index=False)

print("Manual review cases:", len(audit))
print("Saved:", path)

print(
    audit[
        ["message_id", "max_similarity"]
    ].to_string(index=False)
)

print(
    "\nNo messages deleted or automatically classified."
)
